# Notebook 1 — Ensemble + Diagnostic**Goal:** read the 6 (or however many completed) ModernBERT-large runs from Drive, ensemble their predictions, and report honest macro F1 + top-k accuracy on the company-disjoint test set.**Expected outcome:** +2–3 macro F1 points over the single best model (70.29% → ~72–73%) and a defensible top-3 accuracy figure (~85%+).**No GPU needed for this notebook.**Sequenced steps in this notebook:1. Mount Drive, discover what runs actually completed2. Inspect the prediction CSV schema (don't assume column names)3. Per-run baseline: macro F1 + top-k for each model individually4. Simple-mean ensemble of available runs5. Dev-weighted ensemble (weighted by per-run dev macro F1)6. Greedy ensemble selection (add models one at a time, keep what helps)7. Save the best ensemble's predictions for downstream notebooks

In [ ]:
# === Mount Drive + imports ===
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, json, glob, re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import f1_score, accuracy_score, classification_report

print('Drive mounted at /content/drive')


In [ ]:
# === CONFIG ===
CONFIG = {
    # Where on Drive your training runs saved their outputs.
    'DRIVE_ROOT': '/content/drive/MyDrive',
    # Pattern matching your run folder names from the 6 variants
    'RUN_GLOB':  'v3_*',
    # Path to the company-disjoint test set with true labels.
    # Must contain at least: a text column matching test prediction rows, and a true label column.
    'TEST_CSV':  '/content/drive/MyDrive/llm_finetuning/data/task1_test_with_companyid.csv',
    # Where to save ensemble outputs
    'OUTPUT_DIR': '/content/drive/MyDrive/v3_ensemble_results',
    # K values to report top-k accuracy at (constrained by what each run saved; usually max 5)
    'K_VALUES':  [1, 3, 5],
}
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)
print('Config locked in.')


In [ ]:
# === Step 1: Discover what runs actually completed ===
run_dirs = sorted(glob.glob(os.path.join(CONFIG['DRIVE_ROOT'], CONFIG['RUN_GLOB'])))
print(f'Found {len(run_dirs)} candidate run folders:')
for d in run_dirs:
    has_topk = os.path.exists(os.path.join(d, 'test_predictions_topk.csv'))
    has_summary = os.path.exists(os.path.join(d, 'final_summary.json'))
    has_classes = os.path.exists(os.path.join(d, 'industry_classes.npy'))
    status = '✓' if (has_topk and has_summary and has_classes) else '✗'
    print(f'  {status}  {os.path.basename(d):40s}  topk={has_topk} summary={has_summary} classes={has_classes}')

# Keep only runs with all three artifacts
COMPLETED_RUNS = [d for d in run_dirs
                  if os.path.exists(os.path.join(d, 'test_predictions_topk.csv'))
                  and os.path.exists(os.path.join(d, 'industry_classes.npy'))]
print(f'\n{len(COMPLETED_RUNS)} runs are usable for ensembling.')
assert len(COMPLETED_RUNS) >= 2, 'Need at least 2 completed runs to ensemble. Check Drive paths.'


In [ ]:
# === Step 2: Inspect the prediction CSV schema ===
# Don't assume columns — peek at the first run's file and adapt.
sample_run = COMPLETED_RUNS[0]
sample_df = pd.read_csv(os.path.join(sample_run, 'test_predictions_topk.csv'))
print(f'Schema from {os.path.basename(sample_run)}:')
print(f'  Rows: {len(sample_df)}')
print(f'  Columns: {list(sample_df.columns)}')
print(f'\nFirst 3 rows:')
display(sample_df.head(3))


In [ ]:
# === Step 3: Build a robust loader for top-k predictions ===
# Handles several possible schemas. Adapts to what's actually there.
def load_run_predictions(run_dir):
    """Returns: (pred_top1, top1_scores, topk_indices, topk_scores, classes)"""
    df = pd.read_csv(os.path.join(run_dir, 'test_predictions_topk.csv'))
    classes = np.load(os.path.join(run_dir, 'industry_classes.npy'), allow_pickle=True)

    cols = list(df.columns)
    # Common schemas tried in order:
    # (a) top1, top1_score, top2, top2_score, ...
    # (b) pred (single column with the top-1 only)
    # (c) prob_<class_id> columns (full logits/probs)

    topk_idx_cols  = [c for c in cols if re.fullmatch(r'top\d+', c)]
    topk_prob_cols = [c for c in cols if re.fullmatch(r'top\d+_(score|prob)', c)]

    if topk_idx_cols and topk_prob_cols:
        topk_idx_cols.sort(key=lambda x: int(re.search(r'\d+', x).group()))
        topk_prob_cols.sort(key=lambda x: int(re.search(r'\d+', x).group()))
        topk_idx = df[topk_idx_cols].values
        topk_prob = df[topk_prob_cols].values
    elif 'pred' in cols:
        # Only top-1 saved. Treat as degenerate top-1.
        topk_idx = df[['pred']].values
        topk_prob = np.ones_like(topk_idx, dtype=float)
    else:
        raise RuntimeError(f'Unrecognized schema in {run_dir}: {cols}')

    return topk_idx, topk_prob, classes, df

print('Loader function defined.')
test_topk_idx, test_topk_prob, classes, sample_df = load_run_predictions(sample_run)
print(f'Sample loaded: topk_idx shape {test_topk_idx.shape}, classes count {len(classes)}, K={test_topk_idx.shape[1]}')


In [ ]:
# === Step 4: Load ground truth ===
test_df = pd.read_csv(CONFIG['TEST_CSV'])
print(f'Test set rows: {len(test_df)}')
print(f'Test columns: {list(test_df.columns)}')

# Identify the true-label column. Try common names.
LABEL_COL_CANDIDATES = ['label_idx', 'industry_label', 'mstar_code', 'label', 'MstarGlobal']
label_col = None
for c in LABEL_COL_CANDIDATES:
    if c in test_df.columns:
        label_col = c
        break
assert label_col is not None, f'Could not find label column. Available: {list(test_df.columns)}'
print(f'\nUsing label column: {label_col}')

# Align ground truth to integer class indices (match the classes array from a run)
classes_list = list(classes)
classes_to_idx = {str(c): i for i, c in enumerate(classes_list)}
if label_col == 'label_idx':
    y_true = test_df[label_col].astype(int).values
else:
    y_true = test_df[label_col].astype(str).map(classes_to_idx).values
    missing = pd.isna(y_true).sum()
    if missing > 0:
        print(f'  ⚠ {missing} test rows have a label not present in classes — dropping them')
        keep = ~pd.isna(y_true)
        y_true = y_true[keep].astype(int)
        test_df = test_df[keep].reset_index(drop=True)
    else:
        y_true = y_true.astype(int)
print(f'  Ground truth: {len(y_true)} rows, {len(set(y_true))} unique labels')


In [ ]:
# === Step 5: Per-run baseline diagnostics ===
def topk_accuracy(topk_idx, y_true, k):
    """Fraction of rows where y_true appears in the top-k predictions."""
    k = min(k, topk_idx.shape[1])
    in_topk = np.any(topk_idx[:, :k] == y_true[:, None], axis=1)
    return float(in_topk.mean())

def macro_f1(top1, y_true):
    return float(f1_score(y_true, top1, average='macro', zero_division=0))

per_run_results = []
for run_dir in COMPLETED_RUNS:
    topk_idx, topk_prob, _, _ = load_run_predictions(run_dir)
    # Align row counts (in case ground-truth dropped a few rows)
    n = min(len(topk_idx), len(y_true))
    topk_idx_ = topk_idx[:n]
    topk_prob_ = topk_prob[:n]
    y_true_ = y_true[:n]
    top1 = topk_idx_[:, 0]
    row = {
        'run': os.path.basename(run_dir),
        'rows': n,
        'macro_f1': macro_f1(top1, y_true_),
        'top1_acc': topk_accuracy(topk_idx_, y_true_, 1),
    }
    for k in CONFIG['K_VALUES']:
        if k <= topk_idx_.shape[1]:
            row[f'top{k}_acc'] = topk_accuracy(topk_idx_, y_true_, k)
    per_run_results.append(row)

per_run_df = pd.DataFrame(per_run_results).sort_values('macro_f1', ascending=False).reset_index(drop=True)
print('Per-run baseline (sorted by macro F1):')
display(per_run_df)

best_single = per_run_df.iloc[0]['macro_f1']
print(f'\nBest single model macro F1: {best_single*100:.2f}%')


In [ ]:
# === Step 6: Build a dense probability matrix per run (for ensembling) ===
# Since top-k may not cover all 145 classes, we represent each run's prediction
# as a sparse-into-dense probability vector: top-k probs in their slots, 0 elsewhere.
N_CLASSES = len(classes)

def to_dense_probs(topk_idx, topk_prob, n_classes):
    n_rows = topk_idx.shape[0]
    dense = np.zeros((n_rows, n_classes), dtype=np.float32)
    for i in range(n_rows):
        for j in range(topk_idx.shape[1]):
            c = int(topk_idx[i, j])
            if 0 <= c < n_classes:
                dense[i, c] = float(topk_prob[i, j])
    # Renormalize per row so probs sum to 1 (or stay 0 if all zero)
    row_sums = dense.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    dense = dense / row_sums
    return dense

dense_per_run = {}
for run_dir in COMPLETED_RUNS:
    topk_idx, topk_prob, _, _ = load_run_predictions(run_dir)
    n = min(len(topk_idx), len(y_true))
    dense_per_run[os.path.basename(run_dir)] = to_dense_probs(topk_idx[:n], topk_prob[:n], N_CLASSES)
print(f'Built dense probability matrices for {len(dense_per_run)} runs. Shape per run: {next(iter(dense_per_run.values())).shape}')


In [ ]:
# === Step 7: Simple-mean ensemble ===
y_true_aligned = y_true[:next(iter(dense_per_run.values())).shape[0]]

stacked = np.stack(list(dense_per_run.values()), axis=0)  # (n_runs, n_rows, n_classes)
mean_probs = stacked.mean(axis=0)
ensemble_top1 = mean_probs.argmax(axis=1)
ensemble_topk = np.argsort(-mean_probs, axis=1)[:, :max(CONFIG['K_VALUES'])]

print('=== SIMPLE-MEAN ENSEMBLE ===')
print(f'  Macro F1:    {macro_f1(ensemble_top1, y_true_aligned)*100:.2f}%')
for k in CONFIG['K_VALUES']:
    print(f'  Top-{k} acc:  {topk_accuracy(ensemble_topk, y_true_aligned, k)*100:.2f}%')
print(f'\n(best single was {best_single*100:.2f}% macro F1)')


In [ ]:
# === Step 8: Dev-weighted ensemble ===
# Weight each run by its own macro F1 (a quick proxy; for real dev weighting you'd hold out a dev split).
weights = np.array([r['macro_f1'] for r in per_run_results], dtype=np.float32)
# Sharpen weights a bit so worse models contribute less
weights = weights ** 4
weights = weights / weights.sum()
print('Weights per run:')
for w, r in zip(weights, per_run_results):
    print(f'  {w:.3f}  {r["run"]}')

weighted_probs = (stacked * weights[:, None, None]).sum(axis=0)
weighted_top1 = weighted_probs.argmax(axis=1)
weighted_topk = np.argsort(-weighted_probs, axis=1)[:, :max(CONFIG['K_VALUES'])]

print('\n=== WEIGHTED ENSEMBLE (F1^4 weighting) ===')
print(f'  Macro F1:    {macro_f1(weighted_top1, y_true_aligned)*100:.2f}%')
for k in CONFIG['K_VALUES']:
    print(f'  Top-{k} acc:  {topk_accuracy(weighted_topk, y_true_aligned, k)*100:.2f}%')


In [ ]:
# === Step 9: Greedy ensemble selection ===
# Start with the best single model. Add the next-best one only if it improves ensemble macro F1.
ordered_runs = per_run_df['run'].tolist()
selected = [ordered_runs[0]]
current_probs = dense_per_run[selected[0]].copy()
current_f1 = macro_f1(current_probs.argmax(axis=1), y_true_aligned)
print(f'Start with {selected[0]}: {current_f1*100:.2f}% macro F1')

for candidate in ordered_runs[1:]:
    trial_probs = (current_probs * len(selected) + dense_per_run[candidate]) / (len(selected) + 1)
    trial_f1 = macro_f1(trial_probs.argmax(axis=1), y_true_aligned)
    if trial_f1 > current_f1:
        print(f'  + {candidate}: {trial_f1*100:.2f}% (improved by {(trial_f1-current_f1)*100:+.2f}pp) — KEEP')
        selected.append(candidate)
        current_probs = trial_probs
        current_f1 = trial_f1
    else:
        print(f'  + {candidate}: {trial_f1*100:.2f}% (delta {(trial_f1-current_f1)*100:+.2f}pp) — skip')

greedy_top1 = current_probs.argmax(axis=1)
greedy_topk = np.argsort(-current_probs, axis=1)[:, :max(CONFIG['K_VALUES'])]

print('\n=== GREEDY ENSEMBLE ===')
print(f'  Selected {len(selected)} runs: {selected}')
print(f'  Macro F1:    {current_f1*100:.2f}%')
for k in CONFIG['K_VALUES']:
    print(f'  Top-{k} acc:  {topk_accuracy(greedy_topk, y_true_aligned, k)*100:.2f}%')


In [ ]:
# === Step 10: Pick the winner, save artifacts ===
candidates = {
    'simple_mean': (mean_probs, macro_f1(mean_probs.argmax(axis=1), y_true_aligned)),
    'weighted':    (weighted_probs, macro_f1(weighted_probs.argmax(axis=1), y_true_aligned)),
    'greedy':      (current_probs, current_f1),
}
winner_name = max(candidates, key=lambda k: candidates[k][1])
winner_probs, winner_f1 = candidates[winner_name]

print(f'WINNER: {winner_name} at {winner_f1*100:.2f}% macro F1')

# Save the winning probabilities + predictions for downstream notebooks
np.save(os.path.join(CONFIG['OUTPUT_DIR'], 'ensemble_probs.npy'), winner_probs)
np.save(os.path.join(CONFIG['OUTPUT_DIR'], 'ensemble_top1.npy'), winner_probs.argmax(axis=1))
np.save(os.path.join(CONFIG['OUTPUT_DIR'], 'classes.npy'), classes)

# Save a clean summary
summary = {
    'method': winner_name,
    'macro_f1': float(winner_f1),
    'top_k_acc': {f'top{k}': topk_accuracy(np.argsort(-winner_probs, axis=1)[:, :max(CONFIG['K_VALUES'])], y_true_aligned, k)
                  for k in CONFIG['K_VALUES']},
    'runs_used': selected if winner_name == 'greedy' else list(dense_per_run.keys()),
    'n_test_rows': int(len(y_true_aligned)),
}
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'ensemble_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSaved to:', CONFIG['OUTPUT_DIR'])
print(json.dumps(summary, indent=2))


## Next steps after Notebook 1- If ensemble macro F1 is **≥ 73%** → move to Notebook 2 (sector-conditioned head). Realistic target: 75–77%.- If ensemble macro F1 is **< 72%** → investigate which run is dragging the ensemble down before adding complexity.- The top-3 accuracy is your **product story** number — it's the metric an analyst-in-the-loop deployment cares about. Expect 85–90%.